# SupplierGraph on Colab
Run the cells top to bottom. **Stop at the data check (step 4) and read it** before the long steps.
Put in Drive (anywhere, subfolders are searched): this notebook's code zip `suppliergraph_colab.zip`, GLEIF Level 1 + Level 2 zips, `full.jsonl.gz` (OCDS, publication 136), CPV zip, `targets.simple.csv`, the CELEX PDF.
Secrets (key icon, left bar): `HF_TOKEN` (or `ANTHROPIC_API_KEY`), notebook access ON.

In [ ]:
# 1. SETTINGS - edit these
DRIVE_ROOT   = "/content/drive/MyDrive/Colab Notebooks/SupplierGraph"     # searched recursively for the data files and the code zip
OUT_DIR      = DRIVE_ROOT + "/suppliergraph_out"
CODE_ZIP     = "suppliergraph_colab.zip"
GLEIF_COUNTRIES = "DE"                      # comma list, e.g. "DE,AT,FR"; "" = all (needs far more RAM/time)
OCDS_LIMIT   = 20000                        # releases to read; 0 = ALL. Start small (dry run), then set 0
LLM_PROVIDER = "hf"                         # "hf" | "anthropic" | "openai_compat"
LLM_MODEL    = "meta-llama/Llama-3.1-8B-Instruct"                           # REQUIRED for hf: a model id you have verified on huggingface.co/inference/models

In [ ]:
# 2. Mount Drive, unpack code
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import os, glob, zipfile
hits = [p for p in glob.glob(DRIVE_ROOT + "/**/" + CODE_ZIP, recursive=True)]
assert hits, f"{CODE_ZIP} not found under {DRIVE_ROOT}"
os.makedirs("/content/sg", exist_ok=True)
zipfile.ZipFile(hits[0]).extractall("/content/sg")
os.chdir("/content/sg"); os.makedirs(OUT_DIR, exist_ok=True)
print("code at", os.getcwd(), os.listdir("."))

Mounted at /content/drive
code at /content/sg ['mappings', 'docs', 'models', 'docker-compose.yml', 'ui', '.github', 'serve.log', 'sparql', '.env.example', 'pyproject.toml', 'shapes', 'data', '.gitignore', 'config', 'src', 'README.md', 'colab', 'scripts', 'deploy', 'ontology', 'Dockerfile', 'Makefile', 'tests']


In [ ]:
# 3. Install (a few minutes)
!apt-get -qq install -y poppler-utils > /dev/null
!pip -q install -e ".[etl,llm]"
!pip -q install falcon

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for suppliergraph (pyproject.toml) ... done


In [ ]:
# 4. Stage data from Drive into data/raw, then check each source. READ THE OUTPUT.
!python scripts/colab_prepare.py --drive "{DRIVE_ROOT}" --root .
!sg data check

In [ ]:
# 5. Environment + LLM smoke test
import os, sys
sys.path.insert(0, "/content/sg/src")
from google.colab import userdata
for k in ("HF_TOKEN","ANTHROPIC_API_KEY"):
    try: os.environ[k] = userdata.get(k)
    except Exception: pass
os.environ["SG_LLM_PROVIDER"] = LLM_PROVIDER
if LLM_MODEL: os.environ["SG_LLM_MODEL"] = LLM_MODEL
os.environ["SG_OCDS_LIMIT"] = str(OCDS_LIMIT)
from suppliergraph.llm import get_llm
llm = get_llm()
print(type(llm).__name__)

OpenAICompatLLM


The cell above only shows which provider was picked. The first real model call happens in the evaluation step (step 8); if it errors there, check the model id and token.

In [ ]:
# 6. Build: staging DB -> ingest -> entity resolution -> obligations -> knowledge graph -> validation
countries = f"--countries {GLEIF_COUNTRIES}" if GLEIF_COUNTRIES else ""
!sg init-db
!sg ingest {countries}
!sg resolve
!sg extract
!sg build-kg
!sg validate
!sg er-eval

In [ ]:
# 7. Train embeddings (evaluation split, then final model for serving)
!sg train
!sg train --final

In [ ]:
# 8. Evaluation of the question answering (uses the LLM; budget the API calls)
!sg eval

In [ ]:
# ** Run this cell instead of Section 4 and 6-8 if data is already saved in Google Drive
import shutil, os
for sub in ("data", "models"):
    src = f"{OUT_DIR}/{sub}"
    assert os.path.exists(src), f"nothing saved at {src}"
    shutil.copytree(src, f"/content/sg/{sub}", dirs_exist_ok=True)
!ls -la /content/sg/data /content/sg/models/current

/content/sg/data:
total 159244
drwx------  3 root root     4096 Nov 21 21:10 .
drwxr-xr-x 17 root root     4096 Nov 21 22:59 ..
-rw-------  1 root root      233 Nov 21 21:10 build_stats.json
-rw-------  1 root root       88 Nov 21 21:10 er_eval.json
drwx------  2 root root     4096 Nov 21 21:10 eval
-rw-------  1 root root        0 Nov 21 21:10 .gitkeep
-rw-------  1 root root 95725136 Nov 21 21:10 kg.nq
-rw-------  1 root root 67309568 Nov 21 21:10 staging.db

/content/sg/models/current:
total 7016
drwx------ 2 root root    4096 Nov 21 20:41 .
drwx------ 3 root root    4096 Nov 21 20:41 ..
-rw------- 1 root root    9143 Nov 21 20:41 metrics.json
-rw------- 1 root root 6484309 Nov 21 21:07 model.pt
-rw------- 1 root root  673176 Nov 21 21:07 vocab.json


In [ ]:
# 9. Save results to Drive (everything except the raw inputs)
import shutil
for f in os.listdir("data"):
    if f != "raw":
        src = os.path.join("data", f)
        (shutil.copytree(src, f"{OUT_DIR}/data/{f}", dirs_exist_ok=True) if os.path.isdir(src)
         else (os.makedirs(f"{OUT_DIR}/data", exist_ok=True), shutil.copy(src, f"{OUT_DIR}/data/{f}")))
if os.path.exists("models"): shutil.copytree("models", f"{OUT_DIR}/models", dirs_exist_ok=True)
!ls -la "{OUT_DIR}/data" "{OUT_DIR}/models"

In [ ]:
# 10. Serve the app (API + UI) and open it
import subprocess, time
srv = subprocess.Popen(["sg","serve","--host","0.0.0.0","--port","8000"], stdout=open("serve.log","w"), stderr=subprocess.STDOUT)
time.sleep(15); print(open("serve.log").read()[-1500:])
from google.colab import output
output.serve_kernel_port_as_window(8000)

INFO:     Started server process [18757]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
# 10b. Fallback if the window does not open: public tunnel (anyone with the URL can reach the app)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
import subprocess, re, time
t = subprocess.Popen(["./cloudflared","tunnel","--url","http://localhost:8000"], stderr=subprocess.PIPE, text=True)
for line in t.stderr:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m: print(m.group(0)); break

cloudflared: Text file busy
https://possibility-backing-enter-algorithm.trycloudflare.com


In [ ]:
# 11. See the knowledge graph (opens an interactive picture of the best-connected suppliers and what surrounds them)
import sys; sys.path.insert(0, "/content/sg/src")
from suppliergraph.config import get_settings
from suppliergraph.store import open_store
from suppliergraph.kg.explore import render
store = open_store(get_settings())
print("triples:", store.triple_count())
print(render(store, "/content/kg_explorer.html", hops=2, max_nodes=150))
import IPython; IPython.display.display(IPython.display.HTML(filename="/content/kg_explorer.html"))

triples: 400532
{'nodes': 150, 'edges': 135, 'file': '/content/kg_explorer.html'}


In [ ]:
# 12. If the Ask tab shows an error, the server's own message is here
# !tail -40 /content/sg/serve.log

In [ ]:
# import sys; sys.path.insert(0, "/content/sg/src")
# from suppliergraph.config import get_settings
# from suppliergraph.store import open_store
# from suppliergraph.llm import get_llm
# from suppliergraph.rag.text2sparql import Text2Sparql
# store, llm = open_store(get_settings()), get_llm()
# r = Text2Sparql(llm, store).run("Which suppliers are flagged for possible sanction exposure through a parent company?")
# for i, a in enumerate(r.attempts, 1):
#     print(f"--- attempt {i}: rows={a.n_rows} error={a.error}\n{a.query}\n")

--- attempt 1: rows=None error=Name filter ['parent company'] names no entity.
SELECT ?sName WHERE {
  ?f a sg:ExposureFlag ; sg:flaggedEntity ?e ; sg:viaEntity ?p ; sg:hops ?hops .
  ?e sg:legalName ?sName .
  ?p sg:legalName ?pName .
  FILTER(CONTAINS(LCASE(STR(?pName)), "parent company") && ?hops > 0)
} ORDER BY ?sName

--- attempt 2: rows=0 error=None
SELECT ?sName WHERE {
  ?f a sg:ExposureFlag ; sg:flaggedEntity ?e ; sg:viaEntity ?p ; sg:hops ?hops .
  ?e sg:legalName ?sName .
  ?p sg:legalName ?pName .
  FILTER(?hops > 0)
} ORDER BY ?sName

--- attempt 3: rows=None error=Name filter ['parent'] names no entity.
SELECT ?sName WHERE {
  ?f a sg:ExposureFlag ; sg:flaggedEntity ?e ; sg:viaEntity ?p ; sg:hops ?hops .
  ?e sg:legalName ?sName .
  ?p sg:legalName ?pName .
  FILTER(CONTAINS(LCASE(STR(?pName)), "parent"))
} ORDER BY ?sName



In [ ]:
# from suppliergraph.namespaces import SPARQL_PREFIX_BLOCK
# print(store.select(SPARQL_PREFIX_BLOCK + "SELECT ?hops (COUNT(*) AS ?n) WHERE { ?f a sg:ExposureFlag ; sg:hops ?hops } GROUP BY ?hops"))

[{'hops': 0, 'n': 7}]
